# Framework 1/4 — Data Preprocessing & Feature Engineering

> **Thesis:** Toward Actionable Root Cause Analysis: Integrating Causal Discovery and
> Counterfactual XAI for Predictive Maintenance
>
> **Prototype dataset:** NASA C-MAPSS FD001 · **Notebook 1 of 4** · 2026-07-30

The framework has nine modules. This notebook covers **Module 1 (preprocessing) and Module 2 (features + health indicator)**.

All settings come from `pipeline_config.py` in the repo root — change values there, not in the
notebook. All logic lives in `src/causalpdm/framework/`, so the notebook stays a narrative over
tested code rather than a place where logic hides.

## Setup

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

import pipeline_config as cfg
from causalpdm.framework import artifacts

from causalpdm.framework import features, preprocessing
print(f"Dataset: C-MAPSS {cfg.DATASET}")

Dataset: C-MAPSS FD001


## Module 1 — Data understanding & preprocessing

Four things happen here, and the third one is the important design decision.

1. **Drop constant sensors.** Some C-MAPSS channels never change. They carry no information and
   would break the statistical tests used later (you cannot compute a correlation with a constant).
2. **Compute RUL** (Remaining Useful Life) from each engine's run-to-failure trajectory.
3. **Normalise per engine, not per fleet.** Every engine starts with a different unknown amount of
   initial wear. If we standardised across the whole fleet, that engine-to-engine offset would
   look like a shared cause of everything. Standardising each engine against itself removes it.
4. **Produce two versions of the data** — explained below.

In [2]:
prep = preprocessing.run(cfg.DATASET, detrend=cfg.REMOVE_TREND_FOR_DISCOVERY)

print(f"Rows              : {prep['n_rows']:,}")
print(f"Engines           : {prep['n_units']}")
print(f"Missing values    : {prep['missing_share']:.1%}")
print(f"Dropped constants : {prep['dropped_constant']}")
print(f"Remaining channels: {len(prep['feature_cols'])}")

Rows              : 20,631
Engines           : 100
Missing values    : 0.0%
Dropped constants : ['setting_3', 's_1', 's_5', 's_10', 's_16', 's_18', 's_19']
Remaining channels: 17


### Why the pipeline keeps two versions of the same data

This is the subtlest decision in the whole framework, so it is worth being explicit.

Engine degradation is a slow trend that pushes **every** sensor in the same direction at once. In
causal language it is an *unobserved common cause*. That creates a conflict:

| | Trend needed? | Why |
|---|---|---|
| Causal discovery | **No** — remove it | A shared hidden driver makes every sensor look connected to every other one. PCMCI also assumes the data is stationary, and a trend is by definition not stationary. |
| Health indicator, SCM, simulation | **Yes** — keep it | The trend *is* the degradation signal. Remove it and there is no failure left to explain. |

So Module 1 returns both: `normalised` (trend intact) and `detrended` (trend removed). Each later
stage uses whichever is correct for it, and says so.

In [3]:
sample_engine = prep["normalised"]["unit"].iloc[0]
n = prep["normalised"][prep["normalised"]["unit"] == sample_engine]
d = prep["detrended"][prep["detrended"]["unit"] == sample_engine]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), sharey=True)
for sensor in ["s_2", "s_4", "s_11"]:
    axes[0].plot(n["cycle"], n[sensor], lw=1.1, label=sensor)
    axes[1].plot(d["cycle"], d[sensor], lw=1.1, label=sensor)
axes[0].set_title(f"'normalised' — trend intact (engine {sample_engine})")
axes[1].set_title("'detrended' — trend removed, used only for discovery")
for ax in axes:
    ax.set_xlabel("cycle"); ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

### Choosing the causal variables

C-MAPSS has 21 sensors, but several measure almost the same physical quantity — `s_8` is fan speed
and `s_13` is *corrected* fan speed, for example. Near-duplicate variables are the worst case for
causal discovery: they make everything look connected to everything, which both slows the search
enormously and produces a graph too dense to interpret.

`pipeline_config.py` therefore lists one sensor per physical measurement location.

In [4]:
if cfg.CAUSAL_VARIABLES:
    variables = [v for v in cfg.CAUSAL_VARIABLES if v in prep["feature_cols"]]
else:
    variables = prep["feature_cols"]

print(f"Using {len(variables)} causal variables: {variables}")

corr = prep["normalised"][variables].corr().abs()
upper = corr.values[np.triu_indices_from(corr, 1)]
print(f"\nMean absolute correlation between them: {upper.mean():.2f}")
print(f"Pairs above 0.85 (near-duplicates)     : {(upper > 0.85).sum()}")

Using 11 causal variables: ['s_2', 's_3', 's_4', 's_7', 's_8', 's_11', 's_12', 's_15', 's_17', 's_20', 's_21']

Mean absolute correlation between them: 0.65
Pairs above 0.85 (near-duplicates)     : 0


## Module 2 — Feature engineering & health indicator

Two different kinds of derived variable get built here, and keeping them separate is a
methodological requirement, not a style choice.

**Safe to use as causal variables:** rolling mean, rolling standard deviation, slope, first and
second derivatives. Each is a transform of *one* channel, so it is still a legitimate measurement.

**NOT safe as a causal variable:** the health indicator, which combines *many* sensors into one
score. If a variable is a deterministic function of other variables in the graph, it breaks the
**faithfulness** assumption that PCMCI depends on — the algorithm would find impossible
independence patterns and produce nonsense. Worse, if the composite were fitted against RUL, the
label would leak into the graph and every "explanation" would just be restating the target.

So the health indicator is built as a **readout function**: fitted once, then applied to whatever
sensor values it is handed — real, intervened, or counterfactual. It never enters the graph.

In [5]:
feat = features.run(prep["normalised"], variables)

print(f"Engineered feature columns: {len(feat['engineered_cols'])}")
print(f"Health indicator captures : {feat['health_indicator_meta']['explained_variance_ratio']:.0%} of sensor variation")
print(f"Excluded from causal graph : {feat['health_indicator_meta']['excluded_from_causal_graph']}")
print(f"Failure threshold          : {feat['failure_threshold']:.3f}")
print(f"Engines reaching that state: {feat['n_engines_with_onset']} / {prep['n_units']}")

Engineered feature columns: 55
Health indicator captures : 69% of sensor variation
Excluded from causal graph : True
Failure threshold          : 0.645
Engines reaching that state: 100 / 100


### Defining the failure event

Root cause analysis has to explain *something*. C-MAPSS gives no per-engine fault label, so the
event has to be defined — here, the first cycle at which the health indicator crosses a
fleet-level threshold.

Being explicit about this matters: every root cause result downstream is conditional on this
definition of "what went wrong".

In [6]:
featured = feat["featured"]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))

for unit in sorted(featured["unit"].unique())[:12]:
    e = featured[featured["unit"] == unit]
    axes[0].plot(e["cycle"], e["health_indicator"], lw=0.9, alpha=0.75)
axes[0].axhline(feat["failure_threshold"], color="crimson", ls="--", lw=1.3, label="failure threshold")
axes[0].set_xlabel("cycle"); axes[0].set_ylabel("health loss")
axes[0].set_title("Health indicator rises as engines degrade (12 engines)")
axes[0].legend(fontsize=8, frameon=False)

onset_cycles = featured[featured["onset"] == 1].groupby("unit")["cycle"].first()
lifetimes = featured.groupby("unit")["cycle"].max()
warning = (lifetimes - onset_cycles).dropna()
axes[1].hist(warning, bins=25, color="steelblue")
axes[1].set_xlabel("cycles of warning before failure")
axes[1].set_title(f"Warning time (median {warning.median():.0f} cycles)")

plt.tight_layout(); plt.show()
print(f"The health indicator gives a median of {warning.median():.0f} cycles of advance warning.")

The health indicator gives a median of 18 cycles of advance warning.


## Save for the next notebook

Results are cached to `artifacts/framework/` so notebook 2 can pick up from here without redoing
this work.

In [7]:
artifacts.save("prep", prep)
artifacts.save("feat", feat)
print("Saved: prep, feat")
print("\nNext: 02_causal_discovery_validation — learn the causal graph with PCMCI+.")

Saved: prep, feat

Next: 02_causal_discovery_validation — learn the causal graph with PCMCI+.


## What this notebook established

| Question | Answer |
|---|---|
| How clean is C-MAPSS? | Completely clean — no missing values, no gaps, regular sampling |
| How much preprocessing does it need? | Very little: drop constant channels, normalise per engine |
| What was the key design decision? | Two data versions — detrended for discovery, trended for everything else |
| Which variables enter the causal graph? | 11 physically distinct sensors (near-duplicates excluded) |
| What is deliberately kept out? | The composite health indicator, because it would break faithfulness |